In [1]:
from pathlib import Path
from time import time
import csv
from PIL import Image
import rasterio
import numpy as np
import os
import torch
from torchvision import transforms
import torchvision.transforms.functional as F
import random
import torchvision.models as models
import torch.nn as nn
from tqdm import tqdm_notebook as tqdm
from IPython.display import clear_output
import json
import math
import hashlib
from datetime import datetime, timezone
from uuid import uuid4

In [2]:
# Locate the existing v1.1 download without changing the working directory.
DATASET_ROOT = next((base / relative
    for base in (Path.cwd(), *Path.cwd().parents)
    for relative in (Path("."), Path("runtime/datasets/Sen1Floods11/v1.1"),
                     Path("datasets/Sen1Floods11/v1.1"))
    if (base / relative / "splits/flood_handlabeled/flood_test_data.csv").is_file()
    and (base / relative / "checkpoints").is_dir()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Start the kernel inside the thesis checkout containing runtime/datasets/Sen1Floods11/v1.1.")
DATASET_ROOT = Path(os.path.relpath(DATASET_ROOT, Path.cwd()))
BASEDIR = DATASET_ROOT
SPLIT_DIR = BASEDIR / "splits/flood_handlabeled"
HAND_DIR = BASEDIR / "data/flood_events/HandLabeled"
WEAK_DIR = BASEDIR / "data/flood_events/WeaklyLabeled"
CHECKPOINT_DIR = Path(os.path.relpath(
    DATASET_ROOT.resolve().parents[2] / "experiments/sen1floods11/runs/best_checkpoints", Path.cwd()))


In [3]:
# Select our trained weights while retaining the authors' evaluation protocol.
MODEL_VARIANT = "permanent_water"
OUR_CHECKPOINTS = {
    "hand_labeled": "Sen1Floods11_hand_labeled_epoch051_valIoU0.539782.cp",
    "s1_weak": "Sen1Floods11_s1_weak_epoch001_valIoU0.438605.cp",
    "s2_weak": "Sen1Floods11_s2_weak_epoch078_valIoU0.619410.cp",
    "permanent_water": "Sen1Floods11_permanent_water_epoch157_valIoU0.854816.cp",
}
CHECKPOINT_PATH = CHECKPOINT_DIR / OUR_CHECKPOINTS[MODEL_VARIANT]
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(CHECKPOINT_PATH)
print("Dataset:", DATASET_ROOT)
print("Our selected checkpoint:", CHECKPOINT_PATH)
# Saved outputs are historical until evaluation is rerun.


Dataset: ..\..\datasets\Sen1Floods11\v1.1
Our selected checkpoint: runs\best_checkpoints\Sen1Floods11_permanent_water_epoch157_valIoU0.854816.cp


In [4]:
def getArrFlood(fname):
    with rasterio.open(fname) as source:
        return source.read()


def read_local_pairs(pairs, scale_radar=True):
    data = []
    for image_path, label_path in pairs:
        if not image_path.is_file() or not label_path.is_file():
            raise FileNotFoundError(f"Missing local pair: {image_path} | {label_path}")
        image = np.nan_to_num(getArrFlood(image_path))
        label = getArrFlood(label_path).copy()
        # Explicitly map -1 to 255 before conversion; avoid obsolete uint8 arithmetic.
        label[label == -1] = 255
        label = label.astype(np.uint8)
        if scale_radar:
            image = (np.clip(image, -50, 1) + 50) / 51
        data.append((image, label))
    print("Loaded pairs:", len(data))
    return data


def local_split_pairs(split_name, permanent=False, traditional=False):
    with (SPLIT_DIR / split_name).open(newline="") as stream:
        rows = [row for row in csv.reader(stream) if row]
    pairs = []
    for image_name, label_name in rows:
        if permanent:
            label_name = label_name.replace("_LabelHand", "_JRCWaterHand")
        if traditional:
            image_name = image_name.replace("_S1Hand", "_S1OtsuLabelHand")
        pairs.append((HAND_DIR / ("S1OtsuLabelHand" if traditional else "S1Hand") / image_name,
                      HAND_DIR / ("JRCWaterHand" if permanent else "LabelHand") / label_name))
    return pairs


def load_flood_train_data():
    return read_local_pairs(local_split_pairs("flood_train_data.csv"))


def local_weak_pairs(label_folder, exclude_bolivia=False):
    pairs = []
    for image in sorted((WEAK_DIR / "S1Weak").glob("*_S1Weak.tif")):
        if exclude_bolivia and "Bolivia" in image.name:
            continue
        label = WEAK_DIR / label_folder / image.name.replace("_S1Weak", "_" + label_folder)
        pairs.append((image, label))
    return pairs


def load_weak_flood_train_data():
    return read_local_pairs(local_weak_pairs("S2IndexLabelWeak", exclude_bolivia=True))


def load_weak_flood_S1_train_data():
    return read_local_pairs(local_weak_pairs("S1OtsuLabelWeak"))


def load_flood_valid_data():
    return read_local_pairs(local_split_pairs("flood_valid_data.csv"))


def load_flood_test_data():
    return read_local_pairs(local_split_pairs("flood_test_data.csv"))


def load_flood_test_perm_data():
    return read_local_pairs(local_split_pairs("flood_test_data.csv", permanent=True))


def load_flood_bolivia_test_data():
    return read_local_pairs(local_split_pairs("flood_bolivia_data.csv"))


def load_flood_bolivia_test_perm_data():
    return read_local_pairs(local_split_pairs("flood_bolivia_data.csv", permanent=True))


def load_flood_test_trad_data():
    return read_local_pairs(local_split_pairs("flood_test_data.csv", traditional=True), scale_radar=False)


def load_flood_bolivia_test_trad_data():
    return read_local_pairs(local_split_pairs("flood_bolivia_data.csv", traditional=True), scale_radar=False)


In [5]:
flood_test_perm_data = load_flood_test_perm_data()
flood_test_all_data = load_flood_test_data()

flood_test_bolivia_perm_data = load_flood_bolivia_test_perm_data()
flood_test_bolivia_all_data = load_flood_bolivia_test_data()

Loaded pairs: 90
Loaded pairs: 90
Loaded pairs: 15
Loaded pairs: 15


In [6]:
def ignore_perm_water(all_water, perm_water):
  w = all_water.copy()
  perm_water = (w == 1) * (perm_water == 1)
  w[perm_water] = 255
  return w

def ignore_flood_water(all_water, perm_water):
  w = all_water.copy()
  flood_water = (w == 1) * (perm_water == 0)
  w[flood_water] = 255
  return w

def get_flood_flood_test_data(all_water_data, perm_water_data):
  flood_flood_test_data = []
  for ((awx, awy), (pwx, pwy)) in zip(all_water_data, perm_water_data):
    pwy = ignore_perm_water(awy, pwy)
    if np.sum(awy == 1) > 0 and np.sum(pwy == 1) == 0:
      continue
    flood_flood_test_data.append((awx, pwy))
  return flood_flood_test_data
  

def get_perm_flood_test_data(all_water_data, perm_water_data):
  perm_flood_test_data = []
  for ((awx, awy), (pwx, pwy)) in zip(all_water_data, perm_water_data):
    pwy = ignore_flood_water(awy, pwy)
    if np.sum(awy == 1) > 0 and np.sum(pwy == 1) == 0:
      continue
    perm_flood_test_data.append((awx, pwy))
  return perm_flood_test_data


In [7]:
flood_flood_test_data = get_flood_flood_test_data(flood_test_all_data, flood_test_perm_data)
perm_flood_test_data = get_perm_flood_test_data(flood_test_all_data, flood_test_perm_data)

flood_flood_bolivia_test_data = get_flood_flood_test_data(flood_test_bolivia_all_data, flood_test_bolivia_perm_data)
perm_flood_bolivia_test_data = get_perm_flood_test_data(flood_test_bolivia_all_data, flood_test_bolivia_perm_data)

In [8]:
class InMemoryDataset(torch.utils.data.Dataset):
  
  def __init__(self, data_list, preprocess_func):
    self.data_list = data_list
    self.preprocess_func = preprocess_func
  
  def __getitem__(self, i):
    return self.preprocess_func(self.data_list[i])
  
  def __len__(self):
    return len(self.data_list)


def processAndAugment(data):
  (x,y) = data
  im,label = x.copy(), y.copy()

  # convert to PIL for easier transforms
  im1 = Image.fromarray(im[0])
  im2 = Image.fromarray(im[1])
  label = Image.fromarray(label.squeeze())

  # Get params for random transforms
  i, j, h, w = transforms.RandomCrop.get_params(im1, (256, 256))
  
  im1 = F.crop(im1, i, j, h, w)
  im2 = F.crop(im2, i, j, h, w)
  label = F.crop(label, i, j, h, w)
  if random.random() > 0.5:
    im1 = F.hflip(im1)
    im2 = F.hflip(im2)
    label = F.hflip(label)
  if random.random() > 0.5:
    im1 = F.vflip(im1)
    im2 = F.vflip(im2)
    label = F.vflip(label)
  
  norm = transforms.Normalize([0.6851, 0.5235], [0.0820, 0.1102])
  im = torch.stack([transforms.ToTensor()(im1).squeeze(), transforms.ToTensor()(im2).squeeze()])
  im = norm(im)
  label = transforms.ToTensor()(label).squeeze()
  if torch.sum(label.gt(.003) * label.lt(.004)):
    label *= 255
  label = label.round()

  return im, label


def processTestIm(data):
  (x,y) = data
  im,label = x.copy(), y.copy()
  norm = transforms.Normalize([0.6851, 0.5235], [0.0820, 0.1102])
  #label[0][0][0] = 255
  # convert to PIL for easier transforms
  im_c1 = Image.fromarray(im[0]).resize((512,512))
  im_c2 = Image.fromarray(im[1]).resize((512,512))
  label = Image.fromarray(label.squeeze()).resize((512,512))

  im_c1s = [F.crop(im_c1, 0, 0, 256, 256), F.crop(im_c1, 0, 256, 256, 256),
            F.crop(im_c1, 256, 0, 256, 256), F.crop(im_c1, 256, 256, 256, 256)]
  im_c2s = [F.crop(im_c2, 0, 0, 256, 256), F.crop(im_c2, 0, 256, 256, 256),
            F.crop(im_c2, 256, 0, 256, 256), F.crop(im_c2, 256, 256, 256, 256)]
  labels = [F.crop(label, 0, 0, 256, 256), F.crop(label, 0, 256, 256, 256),
            F.crop(label, 256, 0, 256, 256), F.crop(label, 256, 256, 256, 256)]


  ims = [torch.stack((transforms.ToTensor()(x).squeeze(),
                    transforms.ToTensor()(y).squeeze()))
                    for (x,y) in zip(im_c1s, im_c2s)]
  ims = [norm(im) for im in ims]
  ims = torch.stack(ims)
  labels = [(transforms.ToTensor()(label).squeeze()) for label in labels]
  labels = torch.stack(labels)
  if torch.sum(labels.gt(.003) * labels.lt(.004)):
    labels *= 255
  labels = labels.round()
  return ims, labels


In [9]:
test_all_data = flood_test_all_data
test_all_dataset = InMemoryDataset(test_all_data, processTestIm)
test_all_loader = torch.utils.data.DataLoader(test_all_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_all_iter = iter(test_all_loader)

test_flood_data = flood_flood_test_data
test_flood_dataset = InMemoryDataset(test_flood_data, processTestIm)
test_flood_loader = torch.utils.data.DataLoader(test_flood_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_flood_iter = iter(test_flood_loader)

test_perm_data = perm_flood_test_data
test_perm_dataset = InMemoryDataset(test_perm_data, processTestIm)
test_perm_loader = torch.utils.data.DataLoader(test_perm_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_perm_iter = iter(test_perm_loader)

c:\Users\Stefano\Desktop\THESIS\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [10]:
test_bolivia_all_data = flood_test_bolivia_all_data
test_bolivia_all_dataset = InMemoryDataset(test_bolivia_all_data, processTestIm)
test_bolivia_all_loader = torch.utils.data.DataLoader(test_bolivia_all_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_bolivia_all_iter = iter(test_bolivia_all_loader)

test_bolivia_flood_data = flood_flood_bolivia_test_data
test_bolivia_flood_dataset = InMemoryDataset(test_bolivia_flood_data, processTestIm)
test_bolivia_flood_loader = torch.utils.data.DataLoader(test_bolivia_flood_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_bolivia_flood_iter = iter(test_bolivia_flood_loader)

test_bolivia_perm_data = perm_flood_bolivia_test_data
test_bolivia_perm_dataset = InMemoryDataset(test_bolivia_perm_data, processTestIm)
test_bolivia_perm_loader = torch.utils.data.DataLoader(test_bolivia_perm_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_bolivia_perm_iter = iter(test_bolivia_perm_loader)

In [11]:
net = models.segmentation.fcn_resnet50(weights=None, num_classes=2, weights_backbone=None)
net.backbone.conv1 = nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3,
                               bias=False)
def convertBNtoGN(module, num_groups=16):
  if isinstance(module, torch.nn.modules.batchnorm.BatchNorm2d):
    return nn.GroupNorm(num_groups, module.num_features,
                        eps=module.eps, affine=module.affine)
    if module.affine:
        mod.weight.data = module.weight.data.clone().detach()
        mod.bias.data = module.bias.data.clone().detach()

  for name, child in module.named_children():
      module.add_module(name, convertBNtoGN(child, num_groups=num_groups))

  return module

net = convertBNtoGN(net)
net

FCN(
  (backbone): IntermediateLayerGetter(
    (conv1): Conv2d(2, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): GroupNorm(16, 64, eps=1e-05, affine=True, bias=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): GroupNorm(16, 64, eps=1e-05, affine=True, bias=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): GroupNorm(16, 64, eps=1e-05, affine=True, bias=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): GroupNorm(16, 256, eps=1e-05, affine=True, bias=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): GroupNorm(1

In [12]:
net.load_state_dict(torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True))

<All keys matched successfully>

In [13]:
# Use CPU when this kernel has no available CUDA support.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", DEVICE)

def computeIOU(output, target):
  output = torch.argmax(output, dim=1).flatten() 
  target = target.flatten().to(output.device)
  no_ignore = target.ne(255)
  output = output.masked_select(no_ignore)
  target = target.masked_select(no_ignore)
  intersection = torch.sum(output * target)
  union = torch.sum(target) + torch.sum(output) - intersection
  iou = (intersection + .0000001) / (union + .0000001)
  if iou != iou:
    print("failed, replacing with 0")
    iou = torch.tensor(0).float()
  return iou
  

def computeAccuracy(output, target):
  output = torch.argmax(output, dim=1).flatten() 
  target = target.flatten().to(output.device)
  no_ignore = target.ne(255)
  output = output.masked_select(no_ignore)
  target = target.masked_select(no_ignore)
  correct = torch.sum(output.eq(target))
  return correct.float() / len(target)
  
def truePositives(output, target):
  output = torch.argmax(output, dim=1).flatten() 
  target = target.flatten().to(output.device)
  no_ignore = target.ne(255)
  output = output.masked_select(no_ignore)
  target = target.masked_select(no_ignore)
  correct = torch.sum(output * target)
  return correct

def trueNegatives(output, target):
  output = torch.argmax(output, dim=1).flatten() 
  target = target.flatten().to(output.device)
  no_ignore = target.ne(255)
  output = output.masked_select(no_ignore)
  target = target.masked_select(no_ignore)
  output = (output == 0)
  target = (target == 0)
  correct = torch.sum(output * target)
  return correct

def falsePositives(output, target):
  output = torch.argmax(output, dim=1).flatten() 
  target = target.flatten().to(output.device)
  no_ignore = target.ne(255)
  output = output.masked_select(no_ignore)
  target = target.masked_select(no_ignore)
  output = (output == 1)
  target = (target == 0)
  correct = torch.sum(output * target)
  return correct

def falseNegatives(output, target):
  output = torch.argmax(output, dim=1).flatten() 
  target = target.flatten().to(output.device)
  no_ignore = target.ne(255)
  output = output.masked_select(no_ignore)
  target = target.masked_select(no_ignore)
  output = (output == 0)
  target = (target == 1)
  correct = torch.sum(output * target)
  return correct


def test_loop(test_data_loader, net):
  net = net.eval()
  net = net.to(DEVICE)
  count = 0
  iou = 0
  loss = 0
  accuracy = 0
  tp = 0
  fp = 0
  tn = 0
  fn = 0
  with torch.no_grad():
      for (images, labels) in tqdm(test_data_loader):
          outputs = net(images.to(DEVICE))
          labels = labels.to(DEVICE)
          valid_iou = computeIOU(outputs["out"], labels)
          iou += valid_iou
          accuracy += computeAccuracy(outputs["out"], labels)
          tp +=  truePositives(outputs["out"], labels)
          fp +=  falsePositives(outputs["out"], labels)
          tn +=  trueNegatives(outputs["out"], labels)
          fn +=  falseNegatives(outputs["out"], labels)
          count += 1

  iou = iou / count
  print("Test Mean IOU:", iou)
  print("Total IOU:", (tp.float() / (fn + fp + tp)))
  print("OMISSON:", fn.float() / (fn + tp))
  print("COMMISSON:", fp.float() / (tn + fp))
  print("Test Accuracy:", accuracy / count)

  # Preserve the historical formulas, including their undefined-value behaviour.
  metrics = {
      "mean_iou": float(iou),
      "total_iou": float(tp.float() / (fn + fp + tp)),
      "omission": float(fn.float() / (fn + tp)),
      "commission": float(fp.float() / (tn + fp)),
      "mean_accuracy": float(accuracy / count),
  }
  nonfinite = [name for name, value in metrics.items() if not math.isfinite(value)]
  # JSON null records undefined values without emitting non-standard NaN/Infinity tokens.
  return {
      "metrics": {name: value if math.isfinite(value) else None for name, value in metrics.items()},
      "nonfinite_metrics": nonfinite,
      "evaluated_batches": count,
      "confusion_counts": {"tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)},
  }


Evaluation device: cpu


In [14]:

# Save each completed evaluation under a distinct name; previous runs are retained.
CHECKPOINT_ORIGIN = "mine"

def save_evaluation_results(results, started_at):
    runtime_root = DATASET_ROOT.resolve().parents[2]
    output_dir = (runtime_root / "experiments/sen1floods11/runs/results/evaluations"
                  / CHECKPOINT_ORIGIN / MODEL_VARIANT)
    output_dir.mkdir(parents=True, exist_ok=True)
    completed_at = datetime.now(timezone.utc)
    run_id = completed_at.strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid4().hex[:8]
    output_path = output_dir / f"{run_id}.json"
    record = {
        "schema_version": 1,
        "run_id": run_id,
        "checkpoint_origin": CHECKPOINT_ORIGIN,
        "model_variant": MODEL_VARIANT,
        "checkpoint": CHECKPOINT_PATH.resolve().relative_to(runtime_root).as_posix(),
        "checkpoint_sha256": hashlib.sha256(CHECKPOINT_PATH.read_bytes()).hexdigest(),
        "dataset": "Sen1Floods11", "dataset_version": "v1.1",
        "started_at_utc": started_at,
        "completed_at_utc": completed_at.isoformat(),
        "device": str(DEVICE), "torch_version": str(torch.__version__),
        "protocol": "Historical Test_Models evaluation; one chip per batch; mean chip IoU and total IoU; original ignored-label handling retained",
        "splits": {name: {"path": (SPLIT_DIR / name).resolve().relative_to(runtime_root).as_posix(),
                          "sha256": hashlib.sha256((SPLIT_DIR / name).read_bytes()).hexdigest()}
                   for name in ("flood_test_data.csv", "flood_bolivia_data.csv")},
        "results": results,
    }
    # Exclusive creation prevents accidental replacement of an existing result file.
    with output_path.open("x", encoding="utf-8") as stream:
        json.dump(record, stream, indent=2, allow_nan=False)
        stream.write("\n")
    print("Saved evaluation:", output_path)
    return output_path

# Evaluate the selected checkpoint on all six historical test subsets.
evaluation_started_at = datetime.now(timezone.utc).isoformat()
evaluation_results = {}
for subset_name, loader in (
    ("test_all_water", test_all_loader),
    ("test_flood_water", test_flood_loader),
    ("test_permanent_water", test_perm_loader),
    ("bolivia_all_water", test_bolivia_all_loader),
    ("bolivia_flood_water", test_bolivia_flood_loader),
    ("bolivia_permanent_water", test_bolivia_perm_loader),
):
    print("\n" + subset_name)
    evaluation_results[subset_name] = test_loop(loader, net)
results_json_path = save_evaluation_results(evaluation_results, evaluation_started_at)



test_all_water


C:\Users\Stefano\AppData\Local\Temp\ipykernel_27968\214696661.py:84: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for (images, labels) in tqdm(test_data_loader):


  0%|          | 0/90 [00:00<?, ?it/s]

Test Mean IOU: tensor(0.2046)
Total IOU: tensor(0.4310)
OMISSON: tensor(0.5229)
COMMISSON: tensor(0.0196)
Test Accuracy: tensor(0.8954)

test_flood_water


  0%|          | 0/90 [00:00<?, ?it/s]

...

Test Mean IOU: tensor(0.3556)
Total IOU: tensor(0.6019)
OMISSON: tensor(0.3518)
COMMISSON: tensor(0.0145)
Test Accuracy: tensor(0.9266)

bolivia_flood_water


  0%|          | 0/15 [00:00<?, ?it/s]

Test Mean IOU: tensor(0.2860)
Total IOU: tensor(0.5942)
OMISSON: tensor(0.3570)
COMMISSON: tensor(0.0145)
Test Accuracy: tensor(0.9294)

bolivia_permanent_water


  0%|          | 0/7 [00:00<?, ?it/s]

Test Mean IOU: tensor(0.4074)
Total IOU: tensor(0.4556)
OMISSON: tensor(0.2791)
COMMISSON: tensor(0.0182)
Test Accuracy: tensor(0.9692)
Saved evaluation: D:\UNI\THESIS\runtime\experiments\sen1floods11\runs\results\evaluations\mine\permanent_water\20261005T180411_577932Z_35e503d4.json
